In [ ]:
from cantuccio import cornerplot

import numpy as np
import scipy
import matplotlib.pyplot as plt

import marimo as mo

## Start generating some data

In [ ]:
def get_samples(means, vars, size):
    samples = []
    labels = []

    for i, (mu, var) in enumerate(zip(means, vars)):
        print(f"adding component {i}")
        _samples = scipy.stats.distributions.norm(mu, var).rvs(size)
        samples.append(_samples)
        labels.append(rf"$\mu_{i}$")

    return dict(zip(labels, np.stack(samples, axis=0)))

In [ ]:
mus = [5, 3, 10]

first_chain = get_samples(mus, [2, 1, 5], 1000)

## let's see how they look

In [ ]:
_ = cornerplot(samples=first_chain)
plt.show()

### We can now change the style...

In [ ]:
from cantuccio.core import OFFDIAG_MODES
styles = mo.ui.dropdown(options=OFFDIAG_MODES, value='hexbin+kde')
styles

In [ ]:
_ =cornerplot(samples=first_chain, offdiag_mode=styles.value)
plt.show()

### ... change the KDE estimation method to use `KDEpy`'s `FFTKDE`...

In [ ]:
_fig, _axs =cornerplot(samples=first_chain, offdiag_mode=styles.value, kde_kwargs={"fast": False})
_ =cornerplot(samples=first_chain, offdiag_mode=styles.value, kde_kwargs={"fast": True}, fig=_fig, axes=_axs, colors=["red"])
plt.show()

### ... add the true values...

In [ ]:
truths = dict(zip(first_chain.keys(), mus))

In [ ]:
_ = cornerplot(samples=first_chain, truths=truths)
plt.show()

### ... plot the difference between data and injection ...

In [ ]:
_ = cornerplot(samples=first_chain, truths=truths, plot_delta=True)
plt.show()

### ... add a second chain ...

In [ ]:
second_chain = get_samples(mus, [1.2, 4.1, 3.7], 1000)

In [ ]:
_ = cornerplot(samples=[first_chain, second_chain], truths=truths)
plt.show()

### ... and label them ...

In [ ]:
_ = cornerplot(samples=[first_chain, second_chain], truths=truths, labels=['first chain', 'second chain'])
plt.show()

### If the ticks overlap, we can decrease their number

In [ ]:
_ = cornerplot(samples=[first_chain, second_chain], truths=truths, labels=['first chain', 'second chain'], n_ticks=3)
plt.show()

### Chains do not have to share the same parameters, but we can still plot them together

In [ ]:
_second_chain = second_chain.copy()
_second_chain[r"$\mu_4$"] = _second_chain[r"$\mu_1$"]

_second_chain.pop(r"$\mu_1$")

_truths = truths.copy()
_truths[r"$\mu_4$"] = truths[r"$\mu_1$"]

_ = cornerplot(samples=[first_chain, _second_chain], truths=_truths, labels=['first chain', 'second chain'], n_ticks=3)
plt.show()

### Now let's see how the corner plot looks for higly correlated parameters under the two KDE estimation methods.

In [ ]:
cov = np.array([[1, 0.99], [0.99, 1]])
x, y = np.random.multivariate_normal([0, 0], cov, size=1000).T
samples = {"x": x, "y": y}

In [ ]:
_fig, _axs = cornerplot(samples=samples, offdiag_mode='kde', labels='scipy KDE', )
_ = cornerplot(samples=samples, offdiag_mode='kde', kde_kwargs={"fast": True, "alpha": 0.5}, fig=_fig, axes=_axs, colors="red", labels='KDEpy FFTKDE')
plt.show()